# P&C Insurance Underwriting Guidelines — RAG Demo

This notebook demonstrates a Retrieval-Augmented Generation pipeline over publicly available P&C insurance documents.

**Stack:** PyMuPDF · Cohere Embed v3 · ChromaDB · Cohere Command R+

**Lines of business covered:**
- Commercial Property (FEMA SFIP, CA DOI, Oregon DFR)
- Commercial General Liability (NY ISO CG 00 01, Wisconsin OCI PI-045)
- Personal Auto (NAIC Consumer Guide, Texas DOI Statistical Plan, Florida DFS Toolkit)

---
**Prerequisites** (run once before this notebook):
```bash
pip install -r requirements.txt
python download_data.py   # download source PDFs
python ingest.py          # chunk, embed, load into ChromaDB
```

In [ ]:
import os
from IPython.display import display, Markdown

# ── Set your Cohere API key ───────────────────────────────────────────────
# Option 1: set in shell before launching Jupyter  → export COHERE_API_KEY=...
# Option 2: uncomment and paste key here (do NOT commit to git)
# os.environ["COHERE_API_KEY"] = "your-api-key-here"

if not os.environ.get("COHERE_API_KEY"):
    raise EnvironmentError("Please set COHERE_API_KEY before running the notebook.")

from rag_query import query, format_response, RAGResult
print("Imports OK — ChromaDB collection will be loaded on first query.")

In [ ]:
def ask(question: str, lob_filter: str = None) -> RAGResult:
    """Run a query and pretty-print the result."""
    result = query(question, lob_filter=lob_filter)
    
    # Render answer as Markdown
    md = f"### Q: {result.question}\n\n{result.answer}\n\n---\n**Sources**\n"
    seen = set()
    for c in sorted(result.citations, key=lambda x: x.relevance_score, reverse=True):
        key = (c.source, c.page)
        if key in seen:
            continue
        seen.add(key)
        md += f"- `{c.source}` page {c.page} — _{c.lob}_ (score: {c.relevance_score:.3f})\n"
    
    display(Markdown(md))
    return result

---
## Query 1 — Commercial Property: Frame Construction Eligibility

In [ ]:
r1 = ask(
    "What are the eligibility criteria and coverage conditions for frame construction commercial property?",
    lob_filter="Commercial Property",
)

---
## Query 2 — CGL: Completed Operations Coverage

In [ ]:
r2 = ask(
    "Does a CGL policy cover completed operations claims, and how does the trigger differ between occurrence and claims-made forms?",
    lob_filter="Commercial General Liability",
)

---
## Query 3 — Personal Auto: MVR Disqualifying Violations

In [ ]:
r3 = ask(
    "What driving record violations disqualify an applicant from personal auto coverage, and what is the lookback period?",
    lob_filter="Personal Auto",
)

---
## Query 4 — CGL: Minimum Limits Structure

In [ ]:
r4 = ask(
    "What are the standard per-occurrence and aggregate limits required for a Commercial General Liability policy?",
    lob_filter="Commercial General Liability",
)

---
## Query 5 — Cross-LOB: Flood and Physical Damage Exclusions

In [ ]:
r5 = ask(
    "How is flood exposure handled across commercial property and personal auto policies, "
    "and what alternative coverage options are available to insureds?"
    # No lob_filter — intentionally cross-LOB
)

---
## Inspect Raw Grounded Spans (Cohere citations)

Cohere Command R+ returns exact character-span citations mapping each claim to its source document.

In [ ]:
import json

print(f"Grounded spans in Answer 2 (CGL completed ops): {len(r2.grounded_spans)}")
for span in r2.grounded_spans[:5]:
    print(json.dumps(span, indent=2))

---
## Interactive Query

Type any underwriting question below.

In [ ]:
custom_question = "What factors affect the premium for a personal auto policy?"
ask(custom_question)